# Lab 2: Baselines

Before any distillation, you need to know where you stand. In this lab you will produce three scores
on the same 1,000 test messages:

1. **The teacher**: a 7 billion parameter model answering from a prompt, with no training.
2. **The baseline**: a small encoder trained on the 616 messages people have labelled.
3. **The reference**: the same encoder trained on every available label, to show what plenty of labels would buy.

Each score comes with a 95% interval, and each is saved so later labs can compare against it.

Allow about 20 to 30 minutes. Run the cells in order.

In [ ]:
from kd_course.env import check_environment

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

## 1. The data

`make_splits` cuts Banking77 into the four piles from lesson 2.1. The seed is fixed, so everyone on
the course gets the same piles.

In [ ]:
from kd_course.data import load_banking77, make_splits

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
LABELS = bundle.label_names
N = len(LABELS)

print(f"Types:               {N}")
print(f"Gold training set:   {len(splits.gold_train)}")
print(f"Gold validation set: {len(splits.gold_val)}")
print(f"Unlabelled pool:     {len(splits.pool)}")
print(f"Test set:            {len(splits.test)}")
print(f"Training messages removed as copies of test messages: {splits.removed_duplicates}")

## 2. The teacher

The teacher is given the list of 77 types and one message, and asked to reply with one type.
It is not trained on anything. A reply that does not name exactly one type counts as wrong.

In [ ]:
from kd_course.metrics import summarize
from kd_course.models import TEACHER_ID, load_causal_lm
from kd_course.prompted import classify_with_llm
from kd_course.results import save_result

teacher, teacher_tokenizer = load_causal_lm(TEACHER_ID)
teacher_pred = classify_with_llm(teacher, teacher_tokenizer, splits.test.texts, LABELS)

teacher_scores = summarize(splits.test.labels, teacher_pred, N)
save_result("teacher_prompted", teacher_scores, teacher_pred)
print(teacher_scores)

In [ ]:
import gc

import torch

del teacher
gc.collect()
torch.cuda.empty_cache()

## 3. The baseline

A small encoder (ModernBERT-base, about 150 million weights) is trained on the 616 gold messages.
The validation set decides which epoch to keep. The test set is touched once, at the end.

In [ ]:
from kd_course.classifier import STUDENT_ENCODER_ID, predict, train_classifier

baseline = train_classifier(
    STUDENT_ENCODER_ID, splits.gold_train, splits.gold_val, num_labels=N, epochs=20
)
for row in baseline.history:
    print(row)

baseline_pred = predict(baseline, splits.test.texts)
baseline_scores = summarize(splits.test.labels, baseline_pred, N)
save_result("baseline_gold_616", baseline_scores, baseline_pred)
print(baseline_scores)

## 4. The reference

Now pretend people had labelled everything. This uses the pool's true labels, which the scenario
says you do not have. It is here only as a reference line: never train a real model in this course
on the pool's true labels.

In [ ]:
from kd_course.data import LabelledSet

del baseline
gc.collect()
torch.cuda.empty_cache()

everything = LabelledSet(
    texts=splits.gold_train.texts + splits.pool.texts,
    labels=splits.gold_train.labels + splits.pool.labels,
)
reference = train_classifier(
    STUDENT_ENCODER_ID, everything, splits.gold_val, num_labels=N, epochs=3
)
reference_pred = predict(reference, splits.test.texts)
reference_scores = summarize(splits.test.labels, reference_pred, N)
save_result("reference_all_labels", reference_scores, reference_pred)
print(reference_scores)

## 5. The three scores side by side

In [ ]:
from kd_course.results import format_results, load_results

print(format_results(load_results()))

## 6. Is the teacher really better than the baseline?

Both were scored on the same messages, so use a paired comparison.

In [ ]:
from kd_course.metrics import accuracy, paired_bootstrap

result = paired_bootstrap(accuracy, splits.test.labels, baseline_pred, teacher_pred)
print(f"Teacher minus baseline: {result.difference * 100:+.1f} points")
print(f"95% interval:           {result.low * 100:+.1f} to {result.high * 100:+.1f} points")
print(f"Share of resamples where the teacher is ahead: {result.probability_b_better:.0%}")

## 7. What did you see?

Double-click this cell and write your answers.

**a. Put the three models in order by accuracy. Did the order match your prediction from lesson 2.1?**

**b. Is the gap between the teacher and the baseline real? Quote the interval that tells you.**

**c. What share of the teacher's replies were unusable? How might you reduce it without touching the test set?**

**d. The reference used about 9,700 human labels and the baseline used 616. If a label costs 20 cents, what did the extra accuracy cost? What would the teacher have to deliver to be the better deal?**

Your three results are now saved in the `results` folder at the top of the repo. Module 3 adds the fourth.